<a href="https://colab.research.google.com/github/dsb4k8/DS-5530_Assignment_2_3_Combined/blob/main/Assignment_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# <h1 align="center">Assignment2</h1>


### Data link Q1: https://app.box.com/s/jm6pw202asu4xd3uypwtry2rqk691y1i
2) >The provided data (link above) contains various details and attributes associated with used cars. The target variable, which is the central focus of analysis, is the price of the used cars, and it is measured in lakhs. The data in this dataset is tabular, with rows and columns, where each row represents a specific used car listing, and each column represents a particular attribute or feature of these cars. Features are Make and model of the car, Location or city of sale, Year of manufacture, Mileage, Odometer (kilometers driven), Fuel type (petrol or diesel), Transmission type (manual or automatic), Number of owners, Engine displacement, Engine horsepower, Number of seats, and Price when the car was new.

Boilerplate imports

In [155]:
import numpy as np
import pandas as pd

### Getting Data

In [156]:
# import requests
# import os
#
# url = "https://app.box.com/s/jm6pw202asu4xd3uypwtry2rqk691y1i"
# target_dir = "Assignment_2/data_raw"
# filename = "train.csv"
#
# # Ensure the directory exists
# os.makedirs(target_dir, exist_ok=True)
#
# # Download and save the file
# response = requests.get(url)
# with open(os.path.join(target_dir, filename), 'wb') as f:
#     f.write(response.content)

## EDA & Preprocessing

In [157]:
df_raw = pd.read_csv('../data_raw/train.csv')
df_raw
# print(df_raw.head())
# print(df_raw.dtypes)

,Unnamed: 0,Name,Location,Year,Kilometers_Driven,Fuel_Type,Transmission,Owner_Type,Mileage,Engine,Power,Seats,New_Price,Price
0,1,Hyundai Creta 1.6 CRDi SX Option,Pune,2015,41000,Diesel,Manual,First,19.67 kmpl,1582 CC,126.2 bhp,5.0,NaN,12.50
1,2,Honda Jazz V,Chennai,2011,46000,Petrol,Manual,First,13 km/kg,1199 CC,88.7 bhp,5.0,8.61 Lakh,4.50
2,3,Maruti Ertiga VDI,Chennai,2012,87000,Diesel,Manual,First,20.77 kmpl,1248 CC,88.76 bhp,7.0,NaN,6.00
3,4,Audi A4 New 2.0 TDI Multitronic,Coimbatore,2013,40670,Diesel,Automatic,Second,15.2 kmpl,1968 CC,140.8 bhp,5.0,NaN,17.74
4,6,Nissan Micra Diesel XV,Jaipur,2013,86999,Diesel,Manual,First,23.08 kmpl,1461 CC,63.1 bhp,5.0,NaN,3.50
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5842,6014,Maruti Swift VDI,Delhi,2014,27365,Diesel,Manual,First,28.4 kmpl,1248 CC,74 bhp,5.0,7.88 Lakh,4.75
5843,6015,Hyundai Xcent 1.1 CRDi S,Jaipur,2015,100000,Diesel,Manual,First,24.4 kmpl,1120 CC,71 bhp,5.0,NaN,4.00
5844,6016,Mahindra Xylo D4 BSIV,Jaipur,2012,55000,Diesel,Manual,Second,14.0 kmpl,2498 CC,112 bhp,8.0,NaN,2.90
5845,6017,Maruti Wagon R VXI,Kolkata,2013,46000,Petrol,Manual,First,18.9 kmpl,998 CC,67.1 bhp,5.0,NaN,2.65


There is an `['Unnamed: 0']` feature which can be dropped. It is basically a redundant index


In [158]:
df_preprocessed = df_raw.copy()
df_preprocessed.drop(columns=['Unnamed: 0'], inplace=True)


# Problems A & B:
##### a) _Look for the missing values in all the columns and either impute them (replace with mean, median, or mode) or drop them. Justify your action for this task._ _(4 points)_

In [159]:
# EDA:

# How many records per column contain nulls?
missing_counts = df_preprocessed.isnull().sum()

# What proportion do the nulls represent per column?
missing_pct = (df_preprocessed.isnull().sum() / len(df_preprocessed)) * 100

# Combining for reporting
missing_report = pd.DataFrame({
    'Missing Count': missing_counts,
    'Percentage (%)': missing_pct
}).sort_values(by='Percentage (%)', ascending=False)

missing_report

,Missing Count,Percentage (%)
New_Price,5032,86.061228
Seats,38,0.649906
Engine,36,0.615700
Power,36,0.615700
Mileage,2,0.034206
Name,0,0.000000
Location,0,0.000000
Year,0,0.000000
Kilometers_Driven,0,0.000000
Fuel_Type,0,0.000000


### A & B Observations/ Considerations, Actions, and Justifications:

1) Problem A) `New_Price` column value is missing in >86% of records. Imputing this much data would basically result in a _fake_ dataset based on a small minority. On the other hand, dropping 86% of the rows would leave almost no data to analyze. For these reasons I think this column should be dropped all together as it does not represent values a significant portion of the dataset

2) Problem B) `Seats`, `Engine`, `Power`, and `Mileage` on the other hand have < 1% null values. I think due to the small size of the missing data, both imputing and dropping are both valid options, but I will opt to impute values to preserve as much data as possible.

    NOTE:

    - For `Engine`, `Power`, and `Milage`, I will impute median column value for null entries because median handles outliers better than mean
    - But for `Seats`, ill have to impute the mode of the column instead because imputing a value like "3.5 seats" doesnt make any sense

3) Problem A) Lastly, there is an unnamed column which should be dropped - its basically a redundant index that starts at 1 instead of 0. will drop along with `New_Price`


In [160]:
# 1/A) Dropping New_Price
# df_preprocessed = df_raw.copy()
# df_preprocessed.columns

# 3/A) Removing unnecessary columns
df_preprocessed.drop(columns=['New_Price'], inplace=True)
# print(df_preprocessed.dtypes)
# df_preprocessed

### BUT FIRST ☝🏽 -  Problem B
##### b) _Remove the units from some of the attributes and only keep the numerical values (for example remove kmpl from “Mileage”, CC from “Engine”, bhp from “Power”, and lakh from “New_price”)._ _(4 points)_
#### Reason I'm doing B before A:
- We should format the data correctly before imputing values from it
- Mileage, Engine (CC displacement amount), Power, etc are currently strings which cannot be worked with numerically.
- Converting these to float values.


In [161]:
#2/B) Removing units and converting Types

#Extracting digit strings and converting them into floats for processing
df_preprocessed['Mileage'] = pd.to_numeric(df_preprocessed['Mileage'].str.extract(r'(\d+\.\d+|\d+)')[0], errors='coerce')
df_preprocessed['Engine'] = pd.to_numeric(df_preprocessed['Engine'].str.extract(r'(\d+)')[0], errors='coerce')
df_preprocessed['Power'] = pd.to_numeric(df_preprocessed['Power'].str.extract(r'(\d+\.\d+|\d+)')[0], errors='coerce')



### Returning To Problem A:
#### To recap
- ##### _imputing median values for nulls within columns with >99% non-null values as it retains more data than_
- ##### _for continuous values, I will take the median value_
- ##### _for discrete values - i.e number of seats, I will impute the mode instead_


In [162]:
#A) Imputing values for type-converted columns

#medians
df_preprocessed['Mileage'] = df_preprocessed['Mileage'].fillna(df_preprocessed['Mileage'].median())
df_preprocessed['Engine'] = df_preprocessed['Engine'].fillna(df_preprocessed['Engine'].median())
df_preprocessed['Power'] = df_preprocessed['Power'].fillna(df_preprocessed['Power'].median())

#mode
if not df_preprocessed['Seats'].mode().empty:
    df_preprocessed['Seats'] = df_preprocessed['Seats'].fillna(df_preprocessed['Seats'].mode()[0])

# df_preprocessed

# verifying nulls have been handled as expected
print("Missing values after processing:")
print(df_preprocessed.isnull().sum())

Missing values after processing:
Name                 0
Location             0
Year                 0
Kilometers_Driven    0
Fuel_Type            0
Transmission         0
Owner_Type           0
Mileage              0
Engine               0
Power                0
Seats                0
Price                0
dtype: int64


## Problem C
C) _Change the categorical variables (“Fuel_Type” and “Transmission”) into numerical one hot encoded value._  _(4 points)._  

In [163]:
# One-Hot Encoding for 'Fuel_Type' and 'Transmission'
df_onehot = pd.get_dummies(df_preprocessed, columns=['Fuel_Type', 'Transmission'], dtype=int)
print(df_onehot.columns)
df_onehot


Index(['Name', 'Location', 'Year', 'Kilometers_Driven', 'Owner_Type',
       'Mileage', 'Engine', 'Power', 'Seats', 'Price', 'Fuel_Type_Diesel',
       'Fuel_Type_Electric', 'Fuel_Type_Petrol', 'Transmission_Automatic',
       'Transmission_Manual'],
      dtype='str')


,Name,Location,Year,Kilometers_Driven,Owner_Type,Mileage,Engine,Power,Seats,Price,Fuel_Type_Diesel,Fuel_Type_Electric,Fuel_Type_Petrol,Transmission_Automatic,Transmission_Manual
0,Hyundai Creta 1.6 CRDi SX Option,Pune,2015,41000,First,19.67,1582.0,126.20,5.0,12.50,1,0,0,0,1
1,Honda Jazz V,Chennai,2011,46000,First,13.00,1199.0,88.70,5.0,4.50,0,0,1,0,1
2,Maruti Ertiga VDI,Chennai,2012,87000,First,20.77,1248.0,88.76,7.0,6.00,1,0,0,0,1
3,Audi A4 New 2.0 TDI Multitronic,Coimbatore,2013,40670,Second,15.20,1968.0,140.80,5.0,17.74,1,0,0,1,0
4,Nissan Micra Diesel XV,Jaipur,2013,86999,First,23.08,1461.0,63.10,5.0,3.50,1,0,0,0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5842,Maruti Swift VDI,Delhi,2014,27365,First,28.40,1248.0,74.00,5.0,4.75,1,0,0,0,1
5843,Hyundai Xcent 1.1 CRDi S,Jaipur,2015,100000,First,24.40,1120.0,71.00,5.0,4.00,1,0,0,0,1
5844,Mahindra Xylo D4 BSIV,Jaipur,2012,55000,Second,14.00,2498.0,112.00,8.0,2.90,1,0,0,0,1
5845,Maruti Wagon R VXI,Kolkata,2013,46000,First,18.90,998.0,67.10,5.0,2.65,0,0,1,0,1


## Problem D
d) _Create one more feature and add this column to the dataset (you can use mutate function in R for this). For example, you can calculate the current age of the car by subtracting “Year” value from the current year.   (4 points)_   

In [164]:
import datetime

#Getting the current year
current_year = datetime.date.today().year

#Creating the 'Car_Age' feature
# This is the Python equivalent of R's mutate()
df_onehot['Car_Age'] = current_year - df_onehot['Year']

# 3. Checking the result
print(df_onehot[['Year', 'Car_Age']].head())

   Year  Car_Age
0  2015       11
1  2011       15
2  2012       14
3  2013       13
4  2013       13


## Problem E
e) _Perform select, filter, rename, mutate, arrange and summarize with group by operations (or their equivalent operations in python) on this dataset._

In [165]:
# 1. Rename 'Kilometers_Driven' to 'Kms_Driven'
df_processed = df_onehot.rename(columns={'Kilometers_Driven': 'Kms_Driven'})

# 2. Mutate: Creating 'Price_USD' feature (1 Lakh = ~1200 USD)
df_processed['Price_USD'] = df_processed['Price'] * 1200

# 3. Filter results to only include cars that are < 15 years old
df_filtered = df_processed[df_processed['Car_Age'] < 15]

# 4. Selecting 'Location', 'Price', 'Car_Age', and 'Kms_Driven'
df_selected = df_filtered[['Location', 'Price', 'Car_Age', 'Kms_Driven']]

# 5. Summarizing with mean and groupBy to find the average price of cars in each city
# Grouping 'Location' and calculating the mean of 'Price'
df_summary = df_selected.groupby('Location')['Price'].mean().reset_index()

# 6. ARRANGE: Sort the summary by Price from highest to lowest
final_output = df_summary.sort_values(by='Price', ascending=False)

# Display the result
print(final_output)

      Location      Price
1    Bangalore  15.681923
3   Coimbatore  15.673883
7        Kochi  11.675860
5    Hyderabad  11.658508
9       Mumbai  10.956096
4        Delhi  10.871422
2      Chennai  10.204326
0    Ahmedabad   9.459593
10        Pune   8.679321
6       Jaipur   7.404963
8      Kolkata   6.360900


In [166]:
#Writing final output to Reports
final_output.to_markdown("../output/summary.md")